Импорты и seed

In [ ]:
import os
import random
import string
import csv
import json
from functools import lru_cache

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

import requests
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Шрифты

In [ ]:
# --- устанавливаем базовые шрифты через apt ---
# Часть пакетов может отсутствовать в конкретном образе Ubuntu.
# Это не критично: просто пропускаем и едем дальше на том, что встало.
font_packages = [
    "fonts-dejavu", "fonts-liberation2", "fonts-noto-core", "fonts-noto-mono",
    "fonts-roboto-unhinted", "fonts-open-sans", "fonts-lato", "fonts-comfortaa",
    "fonts-crosextra-carlito", "fonts-crosextra-caladea", "fonts-freefont-ttf",
    "fonts-cantarell", "fonts-oxygen", "fonts-ubuntu",
]

get_ipython().system("apt-get update -qq")
ok, failed = [], []
for pkg in font_packages:
    ret = os.system(f"apt-get install -qq -y {pkg} > /dev/null 2>&1")
    (ok if ret == 0 else failed).append(pkg)

print(f"встало: {len(ok)} пакетов")
if failed:
    print("не нашлось в репах (пропускаем):", failed)

In [ ]:
# --- резервный список кириллических семейств Google Fonts ---
# Нужен, если metadata-эндпоинт не ответит или изменится.
FALLBACK_CYRILLIC_FAMILIES = [
    "PT Sans", "PT Sans Caption", "PT Sans Narrow", "PT Serif", "PT Serif Caption", "PT Mono",
    "Russo One", "Neucha", "Underdog", "Kurale", "Ruslan Display", "Cuprum", "Actor",
    "Podkova", "Jura", "Play", "Philosopher", "Marck Script", "Yeseva One",
    "Old Standard TT", "EB Garamond", "Droid Serif", "Merriweather", "Alegreya",
    "Playfair Display", "Bitter", "Roboto Slab",
    "Exo 2", "Rubik", "Montserrat", "Oswald", "Nobile", "News Cycle", "Fjalla One",
    "Yanone Kaffeesatz", "El Messiri",
    "Caveat", "Amatic SC",
    "Fantasque Sans Mono", "Oranienbaum", "Passion One", "Anton",
]

# --- пытаемся получить список семейств с кириллицей с fonts.google.com ---
cyrillic_families = None
try:
    headers = {"User-Agent": "Mozilla/5.0 ..."}
    resp = requests.get("https://fonts.google.com/metadata/fonts", headers=headers, timeout=30)
    text = resp.text.lstrip()
    print("status:", resp.status_code, "| начало ответа:", repr(text[:60]))

    # Google отдаёт JSON с защитным префиксом )]}' — его нужно срезать.
    # Режем по длине префикса, а не по первому \n: \n может быть внутри JSON.
    if text.startswith(")]}'"):
        text = text[len(")]}'"):].lstrip("\n")

    meta = json.loads(text)
    cyrillic_families = [
        fam["family"] for fam in meta["familyMetadataList"]
        if "cyrillic" in fam.get("subsets", [])
    ]
    print(f"получили с fonts.google.com: {len(cyrillic_families)} семейств с кириллицей")
except Exception as e:
    print(f"не получилось дёрнуть метаданные ({e!r}), едем на резервном списке")

if not cyrillic_families:
    cyrillic_families = FALLBACK_CYRILLIC_FAMILIES
    print(f"используем резервный список: {len(cyrillic_families)} семейств")

In [ ]:
# --- sparse clone Google Fonts ---
# ВАЖНО: candidate_paths может содержать несуществующие пути.
# В некоторых версиях git sparse-checkout падает, если путь не найден.
def family_to_slug(name):
    return "".join(ch for ch in name.lower() if ch.isalnum())

GFONTS_DIR = "/content/gfonts"

candidate_paths = []
for fam in cyrillic_families:
    slug = family_to_slug(fam)
    for license_dir in ("ofl", "apache", "ufl"):
        candidate_paths.append(f"{license_dir}/{slug}")

get_ipython().system(
    f"git clone --depth 1 --filter=blob:none --sparse -q https://github.com/google/fonts.git {GFONTS_DIR}"
)
paths_arg = " ".join(candidate_paths)
get_ipython().system(f"git -C {GFONTS_DIR} sparse-checkout set {paths_arg}")

n_files = sum(len(files) for _, _, files in os.walk(GFONTS_DIR) if files)
print(f"скачано файлов (включая .ttf/.otf и метаданные): {n_files}")

In [ ]:
!pip install -q fonttools

In [ ]:
# --- поиск шрифтов и проверка покрытия символов ---
from fontTools.ttLib import TTFont, TTCollection

FONT_SEARCH_DIRS = ["/usr/share/fonts", "/usr/local/share/fonts"]
if os.path.isdir(GFONTS_DIR):
    FONT_SEARCH_DIRS.append(GFONTS_DIR)

def find_fonts(dirs):
    """Рекурсивно собирает пути к .ttf/.otf/.ttc."""
    paths = []
    for root in dirs:
        if not os.path.isdir(root):
            continue
        for dp, _, files in os.walk(root):
            for fn in files:
                if fn.lower().endswith((".ttf", ".otf", ".ttc")):
                    paths.append(os.path.join(dp, fn))
    return sorted(set(paths))

@lru_cache(maxsize=None)
def font_cmap(path):
    """Возвращает set кодов символов, которые есть в шрифте. Кэшируем."""
    try:
        tt = TTCollection(path).fonts[0] if path.lower().endswith(".ttc") else TTFont(path, lazy=True)
        cmap = tt.getBestCmap()
        return frozenset(cmap.keys()) if cmap else None
    except Exception:
        return None

def font_covers(path, text, min_coverage=1.0):
    """Проверяет, что шрифт покрывает нужный текст."""
    cmap = font_cmap(path)
    if cmap is None:
        return False
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return True
    covered = sum(1 for c in chars if ord(c) in cmap)
    return covered / len(chars) >= min_coverage

ALL_FONTS = find_fonts(FONT_SEARCH_DIRS)
print(f"нашли {len(ALL_FONTS)} файлов шрифтов")

# Отдельно оставляем только те, что тянут кириллицу.
cyr_fonts = [p for p in ALL_FONTS if font_covers(p, "АБВГДЕЁЖЗИЙ")]
print(f"из них с кириллицей: {len(cyr_fonts)}")

In [ ]:
def get_family_name(path):
    try:
        tt = TTCollection(path).fonts[0] if path.lower().endswith(".ttc") else TTFont(path, lazy=True)
        name_table = tt["name"]
        for name_id in (16, 1):
            rec = name_table.getDebugName(name_id)
            if rec:
                return rec
    except Exception:
        pass
    return None

families = {}
for p in cyr_fonts:
    fam = get_family_name(p) or os.path.basename(p)
    families.setdefault(fam, []).append(p)

print(f"файлов с кириллицей: {len(cyr_fonts)}")
print(f"РЕАЛЬНО уникальных семейств: {len(families)}")
print()
for fam, files in sorted(families.items()):
    print(f"  {fam:35s} — {len(files)} начертан.")

Текстовый корпус

RU/EN слова и фразы, характерные для объявлений/упаковки/интерфейсов, плюс бренды
и отдельно — шаблоны, где цифры встречаются "органически" внутри фразы.

In [ ]:
CORPUS_RU = [
    "Продам",
    "Продаю",
    "Куплю",
    "Обмен",
    "Новый",
    "Новое",
    "Б/у",
    "Отличное состояние",
    "Как новый",
    "Торг уместен",
    "Без торга",
    "Самовывоз",
    "Доставка",
    "Доставка по России",
    "Оригинал",
    "Гарантия",
    "Гарантия 1 год",
    "Комплект",
    "В комплекте",
    "Не работает",
    "Требует ремонта",
    "На запчасти",
    "Срочно",
    "Срочная продажа",
    "Цена снижена",
    "Скидка",
    "Акция",
    "Распродажа",
    "Октябрьские цены",
    "Октябрьские цены на материалы",
    "Цены на материалы",
    "Почему ценят наше оборудование",
    "Окружающий мир",
    "Учебник",
    "Рабочая тетрадь",
    "Тетрадь",
    "Типография",
    "Издательство",
    "Детская литература",
    "Энциклопедия",
    "Атлас",
    "Словарь",
    "Диван",
    "Кровать",
    "Шкаф",
    "Стол",
    "Стул",
    "Кресло",
    "Комод",
    "Тумба",
    "Зеркало",
    "Ковер",
    "Холодильник",
    "Стиральная машина",
    "Посудомоечная машина",
    "Микроволновая печь",
    "Пылесос",
    "Утюг",
    "Фен",
    "Кофемашина",
    "Электрочайник",
    "Мультиварка",
    "Блендер",
    "Миксер",
    "Духовой шкаф",
    "Варочная панель",
    "Кондиционер",
    "Обогреватель",
    "Вентилятор",
    "Увлажнитель воздуха",
    "Телевизор",
    "Смартфон",
    "Планшет",
    "Ноутбук",
    "Компьютер",
    "Монитор",
    "Клавиатура",
    "Мышь",
    "Наушники",
    "Колонки",
    "Зарядное устройство",
    "Powerbank",
    "Внешний аккумулятор",
    "Флешка",
    "Карта памяти",
    "Жесткий диск",
    "Видеокарта",
    "Процессор",
    "Материнская плата",
    "Блок питания",
    "Корпус",
    "Роутер",
    "Модем",
    "Веб-камера",
    "Принтер",
    "Сканер",
    "МФУ",
    "Картридж",
    "Тонер",
    "Велосипед",
    "Самокат",
    "Скейтборд",
    "Ролики",
    "Коньки",
    "Лыжи",
    "Сноуборд",
    "Мяч",
    "Ракетка",
    "Гантели",
    "Штанга",
    "Тренажер",
    "Беговая дорожка",
    "Гиря",
    "Коврик для йоги",
    "Палатка",
    "Спальный мешок",
    "Рюкзак",
    "Термос",
    "Фонарик",
    "Компас",
    "Удочка",
    "Спиннинг",
    "Автомобиль",
    "Мотоцикл",
    "Скутер",
    "Прицеп",
    "Шины",
    "Диски",
    "Аккумулятор",
    "Двигатель",
    "Коробка передач",
    "Автомагнитола",
    "Видеорегистратор",
    "Автокресло",
    "Детская коляска",
    "Манеж",
    "Кроватка",
    "Погремушка",
    "Конструктор",
    "Пазл",
    "Настольная игра",
    "Кукла",
    "Машинка",
    "Робот",
    "Гараж",
    "Дача",
    "Участок",
    "Квартира",
    "Дом",
    "Комната",
    "Офис",
    "Склад",
    "Инструмент",
    "Дрель",
    "Шуруповерт",
    "Перфоратор",
    "Болгарка",
    "Лобзик",
    "Пила",
    "Молоток",
    "Отвертка",
    "Гаечный ключ",
    "Уровень",
    "Рулетка",
    "Стремянка",
    "Лестница",
    "Краска",
    "Кисть",
    "Валик",
    "Шпатель",
    "Обои",
    "Ламинат",
    "Плитка",
    "Линолеум",
    "Двери",
    "Окна",
    "Балконная рама",
    "Радиатор",
    "Счетчик воды",
    "Насос",
    "Фильтр для воды",
    "Септик",
    "Скважина",
    "Насосная станция",
    "Генератор",
    "Сварочный аппарат",
    "Компрессор",
    "Бензопила",
    "Триммер",
    "Газонокосилка",
    "Культиватор",
    "Мотоблок",
    "Теплица",
    "Парник",
    "Семена",
    "Саженцы",
    "Удобрение",
    "Гроза",
    "Куртка",
    "Пальто",
    "Пуховик",
    "Костюм",
    "Платье",
    "Юбка",
    "Брюки",
    "Джинсы",
    "Футболка",
    "Рубашка",
    "Свитер",
    "Кроссовки",
    "Ботинки",
    "Туфли",
    "Сапоги",
    "Сумка",
    "Рюкзак школьный",
    "Часы",
    "Кольцо",
    "Серьги",
    "Цепочка",
    "Браслет",
    "Очки",
    "Солнцезащитные очки",
    "Фотоаппарат",
    "Объектив",
    "Штатив",
    "Вспышка",
    "Гитара",
    "Пианино",
    "Синтезатор",
    "Барабаны",
    "Микрофон",
    "Усилитель",
    "Аквариум",
    "Клетка для животных",
    "Переноска",
    "Корм для собак",
    "Корм для кошек",
    "Лежанка",
    "Поводок",
    "Ошейник",
    "10 скоростных режимов",
    "Мощность двигателя",
    "Объем бака",
    "Вместимость",
    "Технические характеристики",
    "Инструкция по применению",
    "Сделано в России",
    "Серийный номер",
    "Артикул",
    "Модель",
    "Год выпуска",
    "Дата производства",
    "Срок годности",
    "Экологические потребности",
    "Природа",
    "Наши достижения",
    "Отзывы покупателей",
    "Контакты продавца",
    "Номер телефона",
    "Адрес доставки",
    "Способ оплаты",
    "Наличный расчет",
    "Безналичный расчет",
    "Рассрочка",
    "Кредит доступен",
]

In [ ]:
CORPUS_EN = [
    "Made in China",
    "Made in Germany",
    "Made in USA",
    "Made in Japan",
    "Made in Korea",
    "New",
    "Brand New",
    "Used",
    "Like New",
    "Original",
    "Genuine",
    "Warranty",
    "One Year Warranty",
    "For Sale",
    "Sale",
    "Discount",
    "Clearance",
    "Free Shipping",
    "Fast Delivery",
    "Pickup Only",
    "Best Offer",
    "Limited Edition",
    "Special Edition",
    "Premium Quality",
    "High Quality",
    "Waterproof",
    "Shockproof",
    "Durable",
    "Lightweight",
    "Portable",
    "Rechargeable",
    "Battery Included",
    "Fully Working",
    "Not Working",
    "Spare Parts",
    "For Parts Only",
    "Model",
    "Serial Number",
    "Part Number",
    "Product Code",
    "Barcode",
    "Weight",
    "Size",
    "Dimensions",
    "Capacity",
    "Volume",
    "Power",
    "Voltage",
    "Frequency",
    "Environmental Needs",
    "Instructions",
    "User Manual",
    "Quick Start Guide",
    "Safety Information",
    "Do Not Wash",
    "Dry Clean Only",
    "Hand Wash Only",
    "Keep Dry",
    "Fragile",
    "Handle With Care",
    "This Side Up",
    "Photo 1 of 5",
    "Photo 2 of 5",
    "Photo 3 of 5",
    "Photo 4 of 5",
    "Photo 5 of 5",
    "Item Details",
    "Product Description",
    "Condition",
    "Category",
    "Brand",
    "Color",
    "Style",
    "Size Chart",
    "Contact Seller",
    "Ask a Question",
    "Add to Cart",
    "Buy Now",
    "In Stock",
    "Out of Stock",
    "Ships From",
    "Ships To",
    "Return Policy",
    "Customer Reviews",
    "Rating",
    "Verified Purchase",
    "Top Seller",
    "Best Value",
    "Editors Choice",
    "Award Winning",
    "Since 1998",
    "Est 2005",
    "All Rights Reserved",
    "Trademark",
    "Registered",
    "Patent Pending",
    "Version",
    "Update Available",
    "Software Included",
    "Compatible With",
    "Requires Batteries",
    "Batteries Not Included",
    "Assembly Required",
    "Some Assembly Required",
    "Ready to Use",
    "Plug and Play",
    "Wireless",
    "Bluetooth",
    "USB Type C",
    "HDMI",
    "Wi-Fi Enabled",
    "Smart",
    "Automatic",
    "Manual Mode",
    "Eco Mode",
    "Turbo Mode",
    "Night Mode",
    "Water Resistant",
    "Anti Slip",
    "Non Stick",
    "Stainless Steel",
    "Aluminum Alloy",
    "Genuine Leather",
    "Cotton Blend",
    "Organic",
    "Eco Friendly",
    "Recyclable",
    "Biodegradable",
    "Vintage",
    "Retro Style",
    "Classic Design",
    "Modern Design",
    "Minimalist",
    "Heavy Duty",
    "Industrial Grade",
    "Professional Grade",
    "Home Use",
    "Outdoor Use",
    "Indoor Use",
    "All Weather",
    "Four Seasons",
    "Adjustable",
    "Foldable",
    "Collapsible",
    "Stackable",
    "Compact",
    "Extra Large",
    "Extra Small",
    "One Size Fits All",
    "Left Handed",
    "Right Handed",
    "Back to Y2K",
    "Environmental Needs",
    "Equatorial Climate",
    "International Standard",
    "Quality Control Passed",
    "Tested and Approved",
]

In [ ]:
BRANDS = [
    "HYUNDAI",
    "SAMSUNG",
    "XIAOMI",
    "BOSCH",
    "PHILIPS",
    "LG",
    "SONY",
    "APPLE",
    "HUAWEI",
    "TEFAL",
    "STIHL",
    "MAKITA",
    "DEWALT",
    "KARCHER",
    "ELECTROLUX",
    "INDESIT",
    "ARISTON",
    "POLARIS",
    "REDMOND",
    "TOYOTA",
    "NISSAN",
    "HONDA",
    "FORD",
    "VOLKSWAGEN",
    "STYLUS",
    "NATIONAL",
    "CALIPSO",
    "GALAXY S22 ULTRA",
    "IPHONE 14 PRO",
    "SEPTIK",
    "NOVA",
    "UNIVERSAL",
    "STANDARD",
    "EXPRESS",
    "COMFORT",
    "DELUXE",
    "CLASSIC",
    "SPORT",
    "ECO",
    "PRO",
    "MAX",
    "PLUS",
    "MINI",
    "LITE",
    "TURBO",
]

In [ ]:
print(len(CORPUS_RU), "ru фраз |", len(CORPUS_EN), "en фраз |", len(BRANDS), "брендов")

In [ ]:
# --- шаблоны с цифрами и артикулами ---
DIGIT_TEMPLATES_RU = [
    "фото {a} из {b}", "Фото {a} из {b}", "лот №{a}", "Лот №{a}",
    "{a} скоростных режимов", "{a} режимов работы",
    "гарантия {a} год", "гарантия {a} месяцев",
    "артикул {code}", "серийный номер {code}",
    "цена {price} руб", "{price} ₽",
]
DIGIT_TEMPLATES_EN = [
    "Photo {a} of {b}", "Item {a} of {b}", "Model {code}", "Part No {code}",
    "Since {year}", "Est {year}", "Version {a}.{b}", "{a}% OFF",
]

def random_alnum_code(rng):
    """Генерирует артикулы/серийники вида LAC-1820C, WB10429J91ZF16706."""
    letters, digits = string.ascii_uppercase, string.digits
    style = rng.randint(0, 2)
    if style == 0:
        prefix = "".join(rng.choices(letters, k=rng.randint(2, 4)))
        number = "".join(rng.choices(digits, k=rng.randint(3, 5)))
        suffix = "".join(rng.choices(letters, k=rng.randint(0, 1)))
        return f"{prefix}-{number}{suffix}"
    elif style == 1:
        parts = []
        for _ in range(rng.randint(3, 5)):
            pool = digits if rng.random() < 0.6 else letters
            parts.append("".join(rng.choices(pool, k=rng.randint(2, 5))))
        return "".join(parts)
    else:
        prefix = "".join(rng.choices(letters, k=rng.randint(1, 3)))
        number = "".join(rng.choices(digits, k=rng.randint(2, 4)))
        tail = rng.choice(["", "ULTRA", "PRO", "TI", "PLUS", "MAX"])
        return f"{prefix}{number} {tail}".strip()

def sample_text(rng):
    """Сэмплит текст из корпусов, иногда режет как детектор."""
    r = rng.random()
    if r < 0.38:
        text = rng.choice(CORPUS_RU)
    elif r < 0.66:
        text = rng.choice(CORPUS_EN)
    elif r < 0.80:
        text = rng.choice(BRANDS)
    elif r < 0.90:
        text = random_alnum_code(rng)
    else:
        tmpl = rng.choice(DIGIT_TEMPLATES_RU) if rng.random() < 0.5 else rng.choice(DIGIT_TEMPLATES_EN)
        text = tmpl.format(
            a=rng.randint(1, 12), b=rng.randint(1, 12),
            code=random_alnum_code(rng),
            price=rng.choice([990, 1500, 2490, 15990, 799]),
            year=rng.randint(1970, 2024),
        )

    # Имитируем обрезание строки детектором: иногда режем слева/справа.
    if rng.random() < 0.3 and len(text) > 6:
        cut_l = rng.randint(0, len(text) // 4) if rng.random() < 0.5 else 0
        cut_r = rng.randint(0, len(text) // 4) if rng.random() < 0.5 else 0
        if len(text) - cut_r - cut_l >= 3:
            text = text[cut_l: len(text) - cut_r]
    return text.strip()

Рендер и деградация

Два "режима" под то, что видно в реальном тесте:
- `banner` — рекламная плашка / наложенный на фото текст: жирный шрифт, чистый
  или градиентный фон, высокий контраст;
- `photo` — мелкая сфотографированная надпись (шильдик, гравировка, мелкая печать):
  текстурный/шумный фон, сильный блюр/шум/JPEG, часто апскейл с совсем маленького
  разрешения.

In [ ]:
# --- фоны ---
def bg_flat_or_gradient(w, h, rng):
    """Однотонный или градиентный фон."""
    c1 = tuple(rng.randint(0, 255) for _ in range(3))
    if rng.random() < 0.5:
        arr = np.full((h, w, 3), c1, dtype=np.uint8)
        return Image.fromarray(arr)
    c2 = tuple(rng.randint(0, 255) for _ in range(3))
    axis = rng.choice([0, 1])
    t = np.linspace(0, 1, w if axis == 1 else h, dtype=np.float32)
    arr = np.zeros((h, w, 3), dtype=np.float32)
    grid = np.tile(t, (h, 1)) if axis == 1 else np.tile(t.reshape(-1, 1), (1, w))
    for ch in range(3):
        arr[..., ch] = c1[ch] * (1 - grid) + c2[ch] * grid
    return Image.fromarray(arr.astype(np.uint8))

def bg_texture(w, h, rng, np_rng):
    """Дешёвая процедурная текстура: дерево/металл/пластик."""
    base = np.array([rng.randint(30, 220) for _ in range(3)], dtype=np.float32)
    small = np_rng.standard_normal((max(2, h // 8), max(2, w // 8), 1)).astype(np.float32)
    low = cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC).reshape(h, w, 1)
    low = (low - low.min()) / (np.ptp(low) + 1e-6)
    high = np_rng.standard_normal((h, w, 1)).astype(np.float32)
    high = (high - high.min()) / (np.ptp(high) + 1e-6)
    mix = np.repeat(0.7 * low + 0.3 * high, 3, axis=2)
    variation = rng.uniform(30, 90)
    arr = base[None, None, :] + (mix - 0.5) * variation
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def pick_colors(rng, high_contrast_prob=0.75):
    """Выбирает контрастные или близкие цвета текста/фона."""
    bg = tuple(rng.randint(0, 255) for _ in range(3))
    lum = 0.299 * bg[0] + 0.587 * bg[1] + 0.114 * bg[2]
    if rng.random() < high_contrast_prob:
        fg = (245, 245, 245) if lum < 128 else (15, 15, 15)
        fg = tuple(int(np.clip(c + rng.randint(-20, 20), 0, 255)) for c in fg)
    else:
        delta = rng.randint(25, 70)
        if lum >= 128:
            delta = -delta
        fg = tuple(int(np.clip(c + delta, 0, 255)) for c in bg)
    return fg, bg

In [ ]:
# --- рендер строки ---
def render_line(text, font_path, mode, rng, np_rng):
    """
    mode='banner' — рекламная плашка: крупный шрифт, чистый/градиентный фон.
    mode='photo'  — мелкая сфотографированная надпись: текстура, шум, блюр.
    """
    font_size = rng.randint(46, 100) if mode == "banner" else rng.randint(18, 46)
    try:
        font = ImageFont.truetype(font_path, font_size)
    except Exception:
        return None

    # bbox нужен, чтобы подогнать холст под текст.
    tmp = ImageDraw.Draw(Image.new("RGB", (10, 10)))
    try:
        bbox = tmp.textbbox((0, 0), text, font=font)
    except Exception:
        return None
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    if tw <= 0 or th <= 0 or tw > 2000:
        return None

    pad_x = rng.randint(2, 10 if mode == "banner" else 6)
    pad_y = rng.randint(1, 8 if mode == "banner" else 5)
    W, H = tw + 2 * pad_x, th + 2 * pad_y

    fg, bg_color = pick_colors(rng)
    if mode == "photo":
        bg_img = bg_texture(W, H, rng, np_rng) if rng.random() < 0.7 else bg_flat_or_gradient(W, H, rng)
    else:
        bg_img = bg_flat_or_gradient(W, H, rng)

    canvas = bg_img.convert("RGB")
    draw = ImageDraw.Draw(canvas)

    stroke_w, stroke_fill = 0, None
    if mode == "banner" and rng.random() < 0.35:
        stroke_w = rng.randint(1, 3)
        lum = 0.299 * bg_color[0] + 0.587 * bg_color[1] + 0.114 * bg_color[2]
        stroke_fill = (0, 0, 0) if lum > 128 else (255, 255, 255)

    draw.text(
        (pad_x - bbox[0], pad_y - bbox[1]),
        text, font=font, fill=fg,
        stroke_width=stroke_w, stroke_fill=stroke_fill,
    )
    return canvas

In [ ]:
# --- деградация ---
def degrade(img, mode, rng, np_rng, strength=1.0):
    """
    strength — общий множитель силы деградации.
    Для photo: downscale, blur, noise, contrast/jpeg связаны одним level.
    Это лучше, чем независимые монетки: меньше шансов переусердствовать.
    """
    arr = np.array(img).astype(np.float32)

    # Небольшой случайный поворот.
    if rng.random() < 0.25:
        angle = rng.uniform(-4, 4)
        h, w = arr.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        arr = cv2.warpAffine(arr.astype(np.uint8), M, (w, h), borderMode=cv2.BORDER_REPLICATE).astype(np.float32)

    if mode == "photo":
        level = rng.random() * strength

        if level > 0.15:
            scale = 1.0 - level * rng.uniform(0.35, 0.65)
            h, w = arr.shape[:2]
            small = cv2.resize(arr, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
            arr = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

        if level > 0.3:
            sigma = 0.3 + level * 1.0
            k = 3 if sigma < 1.0 else 5
            arr = cv2.GaussianBlur(arr, (k, k), sigmaX=sigma)

        if rng.random() < 0.5:
            noise_sigma = 2 + level * 10
            arr = arr + np_rng.standard_normal(arr.shape) * noise_sigma

        if rng.random() < 0.4:
            arr = arr * rng.uniform(0.6, 1.25) + rng.uniform(-20, 20)
    else:
        if rng.random() < 0.3:
            arr = cv2.GaussianBlur(arr, (3, 3), sigmaX=rng.uniform(0.2, 0.8) * strength)
        if rng.random() < 0.3:
            arr = arr + np_rng.standard_normal(arr.shape) * rng.uniform(1, 6)
        if rng.random() < 0.3:
            arr = arr * rng.uniform(0.8, 1.15)

    arr = np.clip(arr, 0, 255).astype(np.uint8)
    out = Image.fromarray(arr)

    quality = rng.randint(15, 55) if mode == "photo" else rng.randint(40, 85)
    bgr = cv2.cvtColor(np.array(out), cv2.COLOR_RGB2BGR)
    ok, enc = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if ok:
        dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        out = Image.fromarray(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB))
    return out

def is_degenerate(img):
    """
    Страховка от сплошных/битых рендеров.
    Ловим по разбросу яркости: если std < 8, скорее всего картинка без структуры.
    """
    arr = np.asarray(img.convert("L"), dtype=np.float32)
    return arr.std() < 8.0

In [ ]:
# --- генерация одного сэмпла ---
def generate_sample(rng, np_rng, mode_ratio_photo=0.65, degrade_strength=1.0):
    text = sample_text(rng)

    # Если есть кириллица — выбираем только кириллические шрифты,
    # иначе можно взять любой.
    candidates = [p for p in cyr_fonts] if any(ord(c) > 127 for c in text) else ALL_FONTS
    candidates = [p for p in candidates if font_covers(p, text)]
    if not candidates:
        return None

    font_path = rng.choice(candidates)
    mode = "photo" if rng.random() < mode_ratio_photo else "banner"

    try:
        img = render_line(text, font_path, mode, rng, np_rng)
        if img is None or is_degenerate(img):
            return None
        img = degrade(img, mode, rng, np_rng, strength=degrade_strength)
        if is_degenerate(img):
            return None
    except Exception:
        # Среди сотен шрифтов почти гарантированно есть битые/патологические.
        # Не роняем весь прогон, а просто пропускаем комбинацию.
        return None

    label = 1 if rng.random() < 0.5 else 0
    if label:
        img = img.rotate(180)
    return img, label, mode, text

Диагностика

In [1]:
# --- быстрая проверка глазами ---
# Прежде чем гнать 250k сэмплов, смотрим на десяток-другой.
# Здесь же можно подобираем DEGRADE_STRENGTH.
np_rng_diag = np.random.default_rng(0)
rng_diag = random.Random(0)

fig, axes = plt.subplots(4, 5, figsize=(16, 7))
strengths = [0.3, 0.6, 1.0, 1.4]
for row, s in enumerate(strengths):
    for col in range(5):
        res = None
        while res is None:
            res = generate_sample(rng_diag, np_rng_diag, mode_ratio_photo=1.0, degrade_strength=s)
        img, label, mode, text = res
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"strength={s}", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

DEGRADE_STRENGTH = 0.85  # подобрано по картинкам выше

NameError: name 'np' is not defined

Генерация полного датасета

In [ ]:
def build_dataset(n, out_dir, seed, degrade_strength=DEGRADE_STRENGTH):
    """
    Генерирует n сэмплов и сохраняет их в out_dir.
    manifest.csv: image_id,label,mode,text
    """
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)  # отдельный генератор, чтобы шум/текстуры тоже были воспроизводимы
    os.makedirs(out_dir, exist_ok=True)
    manifest_path = os.path.join(out_dir, "manifest.csv")
    made = 0

    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_id", "label", "mode", "text"])
        pbar = tqdm(total=n)
        attempts = 0

        # Ограничиваем число попыток, чтобы не зациклиться на плохих комбинациях.
        while made < n and attempts < n * 20:
            attempts += 1
            res = generate_sample(rng, np_rng, degrade_strength=degrade_strength)
            if res is None:
                continue

            img, label, mode, text = res
            image_id = f"synth_{made:07d}.jpg"
            img.save(os.path.join(out_dir, image_id), quality=92)
            writer.writerow([image_id, label, mode, text])
            made += 1
            pbar.update(1)

        pbar.close()
    print(f"{out_dir}: {made} сэмплов ({attempts} попыток)")

# Отдельные seed для train/val — чтобы val не был слишком похож на train.
build_dataset(n=250_000, out_dir="/content/synth_train", seed=42)
build_dataset(n=8_000, out_dir="/content/synth_val", seed=999)

Dataset и DataLoader

In [ ]:
INPUT_H, INPUT_W = 32, 128

def to_fixed_canvas(img_gray, target_h=INPUT_H, target_w=INPUT_W):
    """
    Приводит кроп к фиксированному размеру:
    - сохраняет аспект по высоте;
    - если ширина больше target_w — режет по центру;
    - если меньше — паддит медианным значением;
    - нормализует в [-1, 1].
    """
    w, h = img_gray.size
    scale = target_h / h
    new_w = max(1, int(round(w * scale)))
    resized = img_gray.resize((new_w, target_h), Image.BILINEAR)
    arr = np.asarray(resized, dtype=np.float32)

    if new_w == target_w:
        canvas = arr
    elif new_w > target_w:
        x0 = (new_w - target_w) // 2
        canvas = arr[:, x0:x0 + target_w]
    else:
        pad = target_w - new_w
        left = pad // 2
        fill = float(np.median(arr)) if arr.size else 0.0
        canvas = np.full((target_h, target_w), fill, dtype=np.float32)
        canvas[:, left:left + new_w] = arr

    canvas = (canvas / 255.0 - 0.5) / 0.5
    return canvas.astype(np.float32)

class OrientationDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        with open(os.path.join(data_dir, "manifest.csv"), encoding="utf-8") as f:
            self.rows = list(csv.DictReader(f))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(os.path.join(self.data_dir, row["image_id"])).convert("L")
        x = torch.from_numpy(to_fixed_canvas(img)).unsqueeze(0)  # (1, H, W)
        y = torch.tensor(float(row["label"]))
        return x, y

train_ds = OrientationDataset("/content/synth_train")
val_ds = OrientationDataset("/content/synth_val")

train_loader = DataLoader(
    train_ds, batch_size=256, shuffle=True,
    num_workers=2, drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=256, shuffle=False,
    num_workers=2, persistent_workers=True,
)
len(train_ds), len(val_ds)

Модель

Вход — фиксированная высота 32px с сохранением аспекта, ширина паддится/режется
до 128px по центру. Цвет не несёт информации об ориентации букв, поэтому берём
grayscale — это втрое дешевле по входным каналам.

In [ ]:
class DWSep(nn.Module):
    """Depthwise-separable блок: depthwise 3x3 + pointwise 1x1."""
    def __init__(self, cin, cout, stride=2):
        super().__init__()
        s = stride if isinstance(stride, tuple) else (stride, stride)
        self.dw = nn.Conv2d(cin, cin, 3, stride=s, padding=1, groups=cin, bias=False)
        self.bn1 = nn.BatchNorm2d(cin)
        self.pw = nn.Conv2d(cin, cout, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act(self.bn1(self.dw(x)))
        x = self.act(self.bn2(self.pw(x)))
        return x

class TinyOrientationNet(nn.Module):
    """
    Лёгкая CNN для 0°/180°.
    Вход: (B, 1, 32, 128).
    Выход: логит (B,).
    """
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.b1 = DWSep(32, 48, stride=2)
        self.b2 = DWSep(48, 64, stride=2)
        self.b3 = DWSep(64, 96, stride=2)
        self.b4 = DWSep(96, 128, stride=(1, 2))  # по высоте уже 4, не сжимаем
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        x = self.b4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x).squeeze(1)

model = TinyOrientationNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"параметров: {n_params:,} (~{n_params * 4 / 1024:.1f} КБ в fp32)")

# sanity-check
with torch.no_grad():
    out = model(torch.randn(4, 1, INPUT_H, INPUT_W).to(device))
out.shape

Обучение

Метрика задачи — `1 - Brier Score`, а не accuracy, поэтому кроме loss/accuracy
считаем Brier по ходу обучения и в конце калибруем вероятности (temperature
scaling), чтобы модель не была самоуверенной на неоднозначных случаях (короткие
обрезанные строки, "симметричные" цифры и т.п.).

In [ ]:
def brier(probs, labels):
    """Brier score: mean((p - y)^2). Чем ниже, тем лучше."""
    return float(np.mean((probs - labels) ** 2))

@torch.no_grad()
def eval_val(model):
    """Собирает логиты и метки на валидации."""
    model.eval()
    logits_all, labels_all = [], []
    for x, y in val_loader:
        x = x.to(device)
        logits_all.append(model(x).cpu().numpy())
        labels_all.append(y.numpy())
    return np.concatenate(logits_all), np.concatenate(labels_all)

EPOCHS = 15
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
eps = 0.02  # label smoothing

history = {"loss": [], "val_brier": [], "val_acc": []}
best_brier = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(device), y.to(device)

        # Label smoothing: 1 -> 0.98, 0 -> 0.02.
        y_s = y * (1 - eps) + (1 - y) * eps

        logits = model(x)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_s)

        opt.zero_grad()
        loss.backward()
        opt.step()
        running += loss.item() * x.size(0)

    sched.step()

    val_logits, val_labels = eval_val(model)
    val_probs = 1 / (1 + np.exp(-val_logits))
    vb = brier(val_probs, val_labels)
    va = float(np.mean((val_probs > 0.5) == (val_labels > 0.5)))

    history["loss"].append(running / len(train_ds))
    history["val_brier"].append(vb)
    history["val_acc"].append(va)

    print(f"[{epoch:02d}] loss={history['loss'][-1]:.4f}  val_acc={va:.4f}  val_brier(T=1)={vb:.4f}")

    # Сохраняем лучшую модель по Brier.
    if vb < best_brier:
        best_brier = vb
        torch.save(model.state_dict(), "/content/model_best.pt")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["loss"]); axes[0].set_title("train loss"); axes[0].set_xlabel("epoch")
axes[1].plot(history["val_brier"], label="brier (T=1)")
axes[1].plot(history["val_acc"], label="accuracy")
axes[1].legend(); axes[1].set_title("val метрики"); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

Калибровка

Подгоняем "резкость" вероятностей под минимум Brier
на валидации — простым grid-search'ем по одному скаляру, без scipy.

In [ ]:
model.load_state_dict(torch.load("/content/model_best.pt"))
val_logits, val_labels = eval_val(model)

best_t, best_b = 1.0, float("inf")
for t in np.linspace(0.2, 5.0, 97):
    probs = 1 / (1 + np.exp(-val_logits / t))
    b = brier(probs, val_labels)
    if b < best_b:
        best_t, best_b = t, b

raw_probs = 1 / (1 + np.exp(-val_logits))
raw_b = brier(raw_probs, val_labels)

print(f"val_brier без калибровки : {raw_b:.4f}")
print(f"val_brier после T={best_t:.3f} : {best_b:.4f}")
print(f"1 - Brier (метрика задачи)  : {1 - best_b:.4f}")

torch.save({"state_dict": model.state_dict(), "temperature": float(best_t),
            "input_h": INPUT_H, "input_w": INPUT_W}, "/content/checkpoint.pt")

Инференс на реальном тесте

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TEST_DIR = "/content/drive/MyDrive/images"

from concurrent.futures import ThreadPoolExecutor

IMG_EXTS = (".png")
test_files = sorted(f for f in os.listdir(TEST_DIR) if f.lower().endswith(IMG_EXTS))
print("файлов:", len(test_files))

def load_one(fn):
    return to_fixed_canvas(Image.open(os.path.join(TEST_DIR, fn)).convert("L"))

X_PATH = "/content/test_X.npy"
if os.path.exists(X_PATH):
    X = np.load(X_PATH)
else:
    with ThreadPoolExecutor(max_workers=32) as ex:
        X = np.stack(list(tqdm(ex.map(load_one, test_files), total=len(test_files))))
    np.save(X_PATH, X)
print(X.shape)

# инференс по готовому массиву
ckpt = torch.load("/content/checkpoint.pt", map_location=device)
model.load_state_dict(ckpt["state_dict"]); model.eval()
temperature = ckpt["temperature"]

probs = []
with torch.no_grad():
    for i in range(0, len(X), 1024):
        xb = torch.from_numpy(X[i:i + 1024]).unsqueeze(1).to(device)
        probs.append(torch.sigmoid(model(xb) / temperature).cpu().numpy())
probs = np.concatenate(probs)

pd.DataFrame({"image_id": [os.path.splitext(f)[0] for f in test_files],
              "p_180": probs}).to_csv("/content/submission.csv", index=False)
print("готово")

In [ ]:
from google.colab import files
files.download("/content/submission.csv")